<a href="https://colab.research.google.com/github/josuemarroquinj/corporate-value-portfolio-risk-analysis/blob/main/notebooks/01_market_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Corporate Value, Portfolio Risk and Market Performance")
print("Phase 2 - Market Data Collection")
print("Research period: 2022-01-01 to 2025-12-31")

Corporate Value, Portfolio Risk and Market Performance
Phase 2 - Market Data Collection
Research period: 2022-01-01 to 2025-12-31


# Phase 2 — Market Data Collection and Preparation

## Corporate Value, Portfolio Risk and Market Performance

**Research project:** Doctoral Research in Corporate Finance

**Empirical period:** January 1, 2022 – December 31, 2025

### Objective

This notebook implements the market-data acquisition and preparation stage of the empirical research.

The analysis covers five U.S. publicly traded corporations:

- Amazon (AMZN)
- Nvidia (NVDA)
- Microsoft (MSFT)
- Exxon Mobil (XOM)
- Berkshire Hathaway Class B (BRK-B)

The S&P 500 Index (^GSPC) is used as the market benchmark, while the 13-week U.S. Treasury Bill (^IRX) is initially used as a proxy for the risk-free rate.

### Main outputs

1. Daily historical prices
2. Adjusted closing prices
3. Trading volume
4. Simple daily returns
5. Logarithmic daily returns
6. Data-quality diagnostics
7. Panel-format dataset
8. Metadata for research reproducibility


In [3]:
# ============================================================
# 1. LIBRARIES
# ============================================================

import os
import warnings

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.6f}"
)

print("Libraries successfully imported.")


Libraries successfully imported.


## 2. Research Design and Parameters

The market-data analysis uses daily observations for the period from January 1, 2022 through December 31, 2025.

The selected companies represent different industries and business models, allowing the subsequent analysis to examine diversification, systematic risk, and portfolio behavior across heterogeneous firms.

The market benchmark is the S&P 500 Index. A short-term U.S. Treasury Bill is initially used as a proxy for the risk-free rate.


In [4]:
# ============================================================
# 2.1 ASSET UNIVERSE
# ============================================================

ASSETS = [
    "AMZN",   # Amazon
    "NVDA",   # Nvidia
    "MSFT",   # Microsoft
    "XOM",    # Exxon Mobil
    "BRK-B"   # Berkshire Hathaway Class B
]

MARKET = "^GSPC"

RISK_FREE = "^IRX"

TICKERS = ASSETS + [
    MARKET,
    RISK_FREE
]

print("Assets:")
for ticker in ASSETS:
    print(f"  - {ticker}")

print(f"\nMarket benchmark: {MARKET}")
print(f"Risk-free proxy: {RISK_FREE}")


Assets:
  - AMZN
  - NVDA
  - MSFT
  - XOM
  - BRK-B

Market benchmark: ^GSPC
Risk-free proxy: ^IRX


In [5]:
# ============================================================
# 2.2 RESEARCH PERIOD
# ============================================================

START_DATE = "2022-01-01"

# yfinance treats the end date as exclusive.
# Therefore, 2026-01-01 allows us to include
# December 31, 2025.

END_DATE = "2026-01-01"

TRADING_DAYS = 252

print(f"Start date: {START_DATE}")
print(f"End date:   {END_DATE}")
print(f"Annualization factor: {TRADING_DAYS} trading days")


Start date: 2022-01-01
End date:   2026-01-01
Annualization factor: 252 trading days


In [6]:
# ============================================================
# 2.3 COMPANY METADATA
# ============================================================

company_info = {
    "AMZN": {
        "Company": "Amazon",
        "Sector": "Consumer Discretionary"
    },

    "NVDA": {
        "Company": "Nvidia",
        "Sector": "Information Technology"
    },

    "MSFT": {
        "Company": "Microsoft",
        "Sector": "Information Technology"
    },

    "XOM": {
        "Company": "Exxon Mobil",
        "Sector": "Energy"
    },

    "BRK-B": {
        "Company": "Berkshire Hathaway",
        "Sector": "Financials / Diversified"
    },

    "^GSPC": {
        "Company": "S&P 500",
        "Sector": "Market Benchmark"
    },

    "^IRX": {
        "Company": "13-Week U.S. Treasury Bill",
        "Sector": "Risk-Free Proxy"
    }
}

company_metadata = pd.DataFrame(
    [
        {
            "Ticker": ticker,
            "Company": info["Company"],
            "Sector": info["Sector"]
        }
        for ticker, info in company_info.items()
    ]
)

company_metadata


,Ticker,Company,Sector
0,AMZN,Amazon,Consumer Discretionary
1,NVDA,Nvidia,Information Technology
2,MSFT,Microsoft,Information Technology
3,XOM,Exxon Mobil,Energy
4,BRK-B,Berkshire Hathaway,Financials / Diversified
5,^GSPC,S&P 500,Market Benchmark
6,^IRX,13-Week U.S. Treasury Bill,Risk-Free Proxy


## 3. Market Data Acquisition

Historical market data are obtained programmatically using the `yfinance` Python library.

The acquisition process is designed to be reproducible: the ticker universe, observation period, frequency, and price-adjustment parameters are explicitly defined in the notebook.


In [7]:
# ============================================================
# 3.1 DOWNLOAD HISTORICAL MARKET DATA
# ============================================================

raw_data = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,
    actions=True,
    progress=True,
    group_by="column"
)

print("\nDownload completed.")
print(f"Rows:    {raw_data.shape[0]}")
print(f"Columns: {raw_data.shape[1]}")


[*********************100%***********************]  7 of 7 completed



Download completed.
Rows:    1003
Columns: 56


In [11]:
# First five observations
raw_data.head()



Price       Adj Close                                                         \
Ticker           AMZN      BRK-B       MSFT      NVDA       XOM        ^GSPC   
Date                                                                           
2022-01-03 170.404495 300.790009 321.856445 30.026146 54.041134 4,796.560059   
2022-01-04 167.522003 308.529999 316.337585 29.197758 56.073849 4,793.540039   
2022-01-05 164.356995 309.920013 304.194000 27.517071 56.771259 4,700.580078   
2022-01-06 163.253998 313.220001 301.790314 28.089262 58.106556 4,696.049805   
2022-01-07 162.554001 319.779999 301.944183 27.161192 58.582844 4,677.029785   

Price                    Close                                            \
Ticker         ^IRX       AMZN      BRK-B       MSFT      NVDA       XOM   
Date                                                                       
2022-01-03 0.053000 170.404495 300.790009 334.750000 30.121000 63.540001   
2022-01-04 0.080000 167.522003 308.529999 329.010010 29.290001 65.930000   
2022-01-05 0.085000 164.356995 309.920013 316.380005 27.604000 66.750000   
2022-01-06 0.090000 163.253998 313.220001 313.880005 28.177999 68.320000   
2022-01-07 0.088000 162.554001 319.779999 314.040009 27.247000 68.879997   

Price                            Dividends                             \
Ticker            ^GSPC     ^IRX      AMZN    BRK-B     MSFT     NVDA   
Date                                                                    
2022-01-03 4,796.560059 0.053000  0.000000 0.000000 0.000000 0.000000   
2022-01-04 4,793.540039 0.080000  0.000000 0.000000 0.000000 0.000000   
2022-01-05 4,700.580078 0.085000  0.000000 0.000000 0.000000 0.000000   
2022-01-06 4,696.049805 0.090000  0.000000 0.000000 0.000000 0.000000   
2022-01-07 4,677.029785 0.088000  0.000000 0.000000 0.000000 0.000000   

Price                                       High                        \
Ticker          XOM    ^GSPC     ^IRX       AMZN      BRK-B       MSFT   
Date                                                                     
2022-01-03 0.000000 0.000000 0.000000 170.703506 301.299988 338.000000   
2022-01-04 0.000000 0.000000 0.000000 171.399994 309.209991 335.200012   
2022-01-05 0.000000 0.000000 0.000000 167.126495 314.480011 326.070007   
2022-01-06 0.000000 0.000000 0.000000 164.800003 314.109985 318.700012   
2022-01-07 0.000000 0.000000 0.000000 165.243500 320.200012 316.500000   

Price                 ...       Low                                  \
Ticker          NVDA  ...      NVDA       XOM        ^GSPC     ^IRX   
Date                  ...                                             
2022-01-03 30.711000  ... 29.785000 61.209999 4,758.169922 0.050000   
2022-01-04 30.468000  ... 28.349001 64.099998 4,774.270020 0.078000   
2022-01-05 29.416000  ... 27.533001 66.480003 4,699.439941 0.080000   
2022-01-06 28.438000  ... 27.065001 67.070000 4,671.259766 0.083000   
2022-01-07 28.422001  ... 27.056999 67.980003 4,662.740234 0.085000   

Price            Open                                                         \
Ticker           AMZN      BRK-B       MSFT      NVDA       XOM        ^GSPC   
Date                                                                           
2022-01-03 167.550003 300.100006 335.350006 29.815001 61.240002 4,778.140137   
2022-01-04 170.438004 301.649994 334.829987 30.277000 64.129997 4,804.509766   
2022-01-05 166.882996 309.869995 325.859985 28.948999 66.500000 4,787.990234   
2022-01-06 163.450500 312.980011 313.149994 27.639999 68.000000 4,693.390137   
2022-01-07 163.839005 315.559998 314.149994 28.141001 68.519997 4,697.660156   

Price               Stock Splits                                               \
Ticker         ^IRX         AMZN    BRK-B     MSFT     NVDA      XOM    ^GSPC   
Date                                                                            
2022-01-03 0.050000     0.000000 0.000000 0.000000 0.000000 0.000000 0.000000   
2022-01-04 0.085000     0.000000 0.000

In [12]:
# Last five observations
raw_data.tail()

Price       Adj Close                                              \
Ticker           AMZN      BRK-B       MSFT       NVDA        XOM   
Date                                                                
2025-12-24 232.380005 501.339996 484.943420 188.380234 116.875412   
2025-12-26 232.520004 498.299988 484.635406 190.297897 116.767570   
2025-12-29 232.070007 501.049988 484.029236 187.990707 118.159645   
2025-12-30 232.529999 503.709991 484.406860 187.311539 118.610596   
2025-12-31 230.820007 502.649994 480.571167 186.272797 117.973373   

Price                                 Close                                   \
Ticker            ^GSPC     ^IRX       AMZN      BRK-B       MSFT       NVDA   
Date                                                                           
2025-12-24 6,932.049805 3.555000 232.380005 501.339996 488.019989 188.610001   
2025-12-26 6,929.939941 3.543000 232.520004 498.299988 487.709991 190.529999   
2025-12-29 6,905.740234 3.538000 232.070007 501.049988 487.100006 188.220001   
2025-12-30 6,896.240234 3.540000 232.529999 503.709991 487.480011 187.539993   
2025-12-31 6,845.500000 3.547000 230.820007 502.649994 483.619995 186.500000   

Price                                       Dividends                    \
Ticker            XOM        ^GSPC     ^IRX      AMZN    BRK-B     MSFT   
Date                                                                      
2025-12-24 119.220001 6,932.049805 3.555000  0.000000 0.000000 0.000000   
2025-12-26 119.110001 6,929.939941 3.543000  0.000000 0.000000 0.000000   
2025-12-29 120.529999 6,905.740234 3.538000  0.000000 0.000000 0.000000   
2025-12-30 120.989998 6,896.240234 3.540000  0.000000 0.000000 0.000000   
2025-12-31 120.339996 6,845.500000 3.547000  0.000000 0.000000 0.000000   

Price                                                High             \
Ticker         NVDA      XOM    ^GSPC     ^IRX       AMZN      BRK-B   
Date                                                                   
2025-12-24 0.000000 0.000000 0.000000 0.000000 232.949997 501.510010   
2025-12-26 0.000000 0.000000 0.000000 0.000000 232.990005 501.559998   
2025-12-29 0.000000 0.000000 0.000000 0.000000 232.600006 501.500000   
2025-12-30 0.000000 0.000000 0.000000 0.000000 232.770004 505.109985   
2025-12-31 0.000000 0.000000 0.000000 0.000000 232.990005 505.890015   

Price                             ...        Low                          \
Ticker           MSFT       NVDA  ...       NVDA        XOM        ^GSPC   
Date                              ...                                      
2025-12-24 489.160004 188.910004  ... 186.589996 119.120003 6,904.910156   
2025-12-26 488.119995 192.690002  ... 188.000000 118.529999 6,921.600098   
2025-12-29 488.350006 188.759995  ... 185.910004 119.400002 6,888.759766   
2025-12-30 489.679993 188.990005  ... 186.929993 120.629997 6,893.470215   
2025-12-31 488.140015 190.559998  ... 186.490005 119.870003 6,844.549805   

Price                     Open                                              \
Ticker         ^IRX       AMZN      BRK-B       MSFT       NVDA        XOM   
Date                                                                         
2025-12-24 3.525000 232.130005 500.290009 485.679993 187.940002 119.330002   
2025-12-26 3.543000 232.039993 500.450012 486.709991 189.919998 118.889999   
2025-12-29 3.530000 231.940002 499.200012 484.859985 187.710007 120.150002   
2025-12-30 3.540000 231.210007 500.980011 485.929993 188.240005 121.099998   
2025-12-31 3.533000 232.910004 503.920013 487.839996 189.570007 121.180000   

Price                            Stock Splits                             \
Ticker            ^GSPC     ^IRX         AMZN    BRK-B     MSFT     NVDA   
Date                                                                       
2025-12-24 6,904.910156 3.553000     0.000000 0.000000 0.000000 0.000000   
2025-12-26 6,936.020020 3.545000     0.000000 0.000000 0.000000 0.000000   
2025-12-29 6,9

In [13]:
print("Available variables:")

for column in raw_data.columns.levels[0]:
    print(f"  - {column}")

Available variables:
  - Adj Close
  - Close
  - Dividends
  - High
  - Low
  - Open
  - Stock Splits
  - Volume


In [14]:
# ============================================================
# 3.2 EXTRACT ADJUSTED CLOSING PRICES
# ============================================================

adj_close = raw_data["Adj Close"].copy()

print("Adjusted closing prices:")
display(adj_close.head())


Adjusted closing prices:


Ticker,AMZN,BRK-B,MSFT,NVDA,XOM,^GSPC,^IRX
Date,,,,,,,
2022-01-03,170.404495,300.790009,321.856445,30.026146,54.041134,"4,796.560059",0.053000
2022-01-04,167.522003,308.529999,316.337585,29.197758,56.073849,"4,793.540039",0.080000
2022-01-05,164.356995,309.920013,304.194000,27.517071,56.771259,"4,700.580078",0.085000
2022-01-06,163.253998,313.220001,301.790314,28.089262,58.106556,"4,696.049805",0.090000
2022-01-07,162.554001,319.779999,301.944183,27.161192,58.582844,"4,677.029785",0.088000


In [15]:
# ============================================================
# 3.3 DATA AVAILABILITY
# ============================================================

availability = pd.DataFrame({
    "Observations": adj_close.count(),
    "Missing": adj_close.isna().sum(),
    "First Observation": [
        adj_close[ticker].first_valid_index()
        for ticker in adj_close.columns
    ],
    "Last Observation": [
        adj_close[ticker].last_valid_index()
        for ticker in adj_close.columns
    ]
})

availability


,Observations,Missing,First Observation,Last Observation
Ticker,,,,
AMZN,1003,0,2022-01-03,2025-12-31
BRK-B,1003,0,2022-01-03,2025-12-31
MSFT,1003,0,2022-01-03,2025-12-31
NVDA,1003,0,2022-01-03,2025-12-31
XOM,1003,0,2022-01-03,2025-12-31
^GSPC,1003,0,2022-01-03,2025-12-31
^IRX,1003,0,2022-01-03,2025-12-31


In [16]:
# ============================================================
# 3.4 MISSING VALUES
# ============================================================

missing_summary = (
    adj_close
    .isna()
    .sum()
    .to_frame("Missing_Observations")
)

missing_summary


,Missing_Observations
Ticker,
AMZN,0
BRK-B,0
MSFT,0
NVDA,0
XOM,0
^GSPC,0
^IRX,0


In [17]:
if missing_summary["Missing_Observations"].sum() == 0:
    print("No missing observations detected in adjusted prices.")
else:
    print("Missing observations detected. Further inspection required.")


No missing observations detected in adjusted prices.
